In [1]:
import sys
print("Executable:", sys.executable)
import fedml, torch
print("fedml:", fedml.__version__)
print("torch:", torch.__version__)

Executable: E:\proekt3\practika_leto\venv-fedml\Scripts\python.exe


E:\proekt3\practika_leto\venv-fedml\Lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.9) doesn't match a supported version!
  warnings.warn(
E:\proekt3\practika_leto\venv-fedml\Lib\site-packages\fedml\computing\scheduler\comm_utils\sys_utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


fedml: 0.9.6
torch: 2.13.0+cpu


In [2]:
config = """common_args:
  training_type: "simulation"
  random_seed: 0
data_args:
  dataset: "mnist"
  data_cache_dir: "./data/mnist"
  partition_method: "homo"
  partition_alpha: 0.5
model_args:
  model: "lr"
train_args:
  federated_optimizer: "FedAvg"
  client_id_list:
  client_num_in_total: 3
  client_num_per_round: 3
  comm_round: 5
  epochs: 1
  batch_size: 10
  client_optimizer: sgd
  learning_rate: 0.03
  weight_decay: 0.001
validation_args:
  frequency_of_the_test: 1
device_args:
  using_gpu: false
  gpu_id: 0
comm_args:
  backend: "sp"
tracking_args:
  enable_wandb: false
  log_file_dir: "./log"
"""
with open("fedml_config_mlp.yaml", "w", encoding="utf-8") as f:
    f.write(config)
print("Конфиг fedml_config_mlp.yaml записан")

Конфиг fedml_config_mlp.yaml записан


In [4]:
import sys, logging, re
import torch
import torch.nn as nn
import fedml
from fedml import FedMLRunner

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)   # ровно 10 выходов — под MNIST
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.fc2(torch.relu(self.fc1(x)))

fedml_acc = []
class AccCollector(logging.Handler):
    def emit(self, record):
        msg = record.getMessage()
        if "_local_test_on_all_clients" in msg and "'test_acc'" in msg:
            m = re.search(r"'test_acc':\s*([0-9.]+)", msg)
            if m:
                fedml_acc.append(float(m.group(1)))

def run_fedml_mlp(config_path):
    sys.argv = ["fedml_mnist.py", "--cf", config_path]
    args = fedml.init()
    device = fedml.device.get_device(args)
    dataset, output_dim = fedml.data.load(args)
    print(">>> output_dim от загрузчика MNIST:", output_dim)
    model = MLP()
    print(">>> выходов у нашей модели:", model.fc2.out_features)
    FedMLRunner(args, device, dataset, model).run()

handler = AccCollector()
logging.getLogger().addHandler(handler)
logging.getLogger().setLevel(logging.INFO)

fedml_acc.clear()
run_fedml_mlp("fedml_config_mlp.yaml")

logging.getLogger().removeHandler(handler)
print("\nFedML (MLP) accuracy по раундам:", [round(a * 100, 2) for a in fedml_acc])

[FedML-Client @device-id-0] [Fri, 24 Jul 2026 11:41:00.304581880] [INFO] [__init__.py:164:init] args.rank = 0, args.worker_num = 3
[FedML-Client @device-id-0] [Fri, 24 Jul 2026 11:41:00.306767463] [INFO] [ml_engine_adapter.py:147:get_torch_device] args = <fedml.arguments.Arguments object at 0x0000018BFE0E9A10>, using_gpu = False, device_id = 0, device_type = cpu
[FedML-Client @device-id-0] [Fri, 24 Jul 2026 11:41:00.308595895] [INFO] [device.py:49:get_device] device = cpu
[FedML-Client @device-id-0] [Fri, 24 Jul 2026 11:41:00.310487508] [INFO] [data_loader.py:21:download_mnist] ./data/mnist\MNIST.zip
[FedML-Client @device-id-0] [Fri, 24 Jul 2026 11:41:00.312160491] [INFO] [data_loader.py:264:load_synthetic_data] load_data. dataset_name = mnist
[FedML-Client @device-id-0] [Fri, 24 Jul 2026 11:41:34.703596353] [INFO] [data_loader.py:123:load_partition_data_mnist] loading data...
[FedML-Client @device-id-0] [Fri, 24 Jul 2026 11:41:39.323784112] [INFO] [data_loader.py:141:load_partition_da

In [5]:
import sys, csv
sys.stdout = sys.__stdout__
sys.stderr = sys.__stderr__

fedml_acc5 = [24.14, 24.94, 26.98, 32.22, 34.89]   # из лога, строки 'test_acc'
print("FedML (MLP, 5 раундов):", fedml_acc5, "-> итог", fedml_acc5[-1], "%")

with open("fedml_mlp5_accuracy.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["round", "test_acc_%"])
    for i, a in enumerate(fedml_acc5, 1):
        w.writerow([i, a])
print("Сохранено: fedml_mlp5_accuracy.csv")